# Surper_GCG Colab 运行面板

这个 notebook 直接调用当前的 `run_pipeline.py` 入口。

当前只讨论已解锁阶段：
1. `Stage 1 = baseline_diagnosis`
2. `Stage 2 = gate_discovery`
3. `eval_calibration / Exp11` 仍然只是可选辅助支线
4. 输出目录写入 `results/pipeline_runs/`


## 1. 挂载 Drive

用 Google Drive 保存仓库副本和 pipeline 输出。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库并安装

把仓库克隆到 `/content`，然后在仓库根目录执行安装。

这个单元是幂等的：如果运行时里已经有仓库，就会跳过 clone。

In [ ]:
%cd /content
import os
if not os.path.exists('/content/Surper_GCG'):
    !git clone https://github.com/awaa-col/Surper_GCG_ResearchWorkFlow.git Surper_GCG
else:
    print('仓库已存在于 /content/Surper_GCG，跳过 clone。')
%cd /content/Surper_GCG
!pip install -e .

## 配置区

设置 Hugging Face 凭据、运行名、模型和共享参数。

In [ ]:
import os

HF_TOKEN = ''
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

RUN_NAME = 'gemma3_12b_main'
ACTIVE_RUN_NAME = RUN_NAME
DRIVE_RUN_DIR = f'/content/drive/MyDrive/{RUN_NAME}'
RESUME = '--resume'  # 新开跑时改成 ''
MODEL = 'google/gemma-3-12b-it'
N_TRAIN = 64
N_EVAL = 2
MAX_NEW_TOKENS = 96


## Stage 目录总览

先列出当前 preset，再确认哪些 stage 已解锁、哪些仍被阻塞。

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py --list-presets

In [ ]:
!python run_pipeline.py --list-stages

## Stage 1. Baseline Diagnosis Dry-Run

从 `baseline_diagnosis` 开始。这是第一个真正的 `12B` 主线科技点。

先做 dry-run，确认 preset、输出目录和生成命令都正确。

In [ ]:
!python run_pipeline.py \
  --preset baseline_diagnosis \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## Stage 1. Baseline Diagnosis

默认主线先跑 `baseline_diagnosis`。

不要把 `eval_calibration` 当成真正的第一关，它只是 review 对齐用的辅助科技。

In [ ]:
!python run_pipeline.py \
  --preset baseline_diagnosis \
  --run-name "$RUN_NAME" \
  $RESUME \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## Stage 2. Gate Discovery 恢复说明

如果 `Stage 1` 已经做完，但 Colab GPU 实例被回收，而你已经把 `gemma3_12b_main` 目录保存到了 Google Drive：

1. 先把 Drive 里的 `gemma3_12b_main` 拉回 Colab 的 `results/pipeline_runs/gemma3_12b_main/`
2. 确保里面至少有 `exp00_diagnosis.json`
3. 然后继续用同一个 `gemma3_12b_main` 跑 `gate_discovery_bootstrap` 并带上 `--resume`

这样 pipeline 会跳过已经存在的 `baseline_diagnosis` 输出，直接继续跑 `gate_discovery`。

In [ ]:
import os
import shutil

src = DRIVE_RUN_DIR
dst = f'/content/Surper_GCG/results/pipeline_runs/{RUN_NAME}'

if not os.path.exists(src):
    raise FileNotFoundError(f'Drive 中未找到目录: {src}')

os.makedirs('/content/Surper_GCG/results/pipeline_runs', exist_ok=True)
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f'已从 Drive 恢复: {src} -> {dst}')
print(f'exp00_diagnosis.json 存在: {os.path.exists(os.path.join(dst, "exp00_diagnosis.json"))}')

## Stage 2. Gate Discovery Dry-Run

在正式跑 gate 之前，先用这个单元确认 preset 和输出路径。

这里仍然只会进入已解锁的 `gate_discovery`，不会碰后面未解锁的阶段。

In [ ]:
!python run_pipeline.py \
  --preset gate_discovery_bootstrap \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## Stage 2. Gate Discovery

只有在 baseline diagnosis 的结果看起来正常之后，才运行这一段。

这会启动第一轮 `12B-first` gate 扫描，但不会提前解锁 cross-layer、detect 或 late-family。

In [ ]:
!python run_pipeline.py \
  --preset gate_discovery_bootstrap \
  --run-name "$RUN_NAME" \
  $RESUME \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## Support Tech. Eval Calibration

只有在你确实想用旧结果生成手工 review pack 时，才运行这一段。

这和从零开始跑 `12B` 主线不是一回事。

In [ ]:
# 可选：只有当你的 Colab 环境里已经有 Exp11 所需的旧结果文件时才运行
!python run_pipeline.py \
  --preset eval_calibration \
  --run-name "${RUN_NAME}_eval_calibration" \
  $RESUME \
  --model "$MODEL"

In [ ]:
print('当前主线阶段：Stage 1 完成后，进入 Stage 2 / gate_discovery。不要提前跑未解锁阶段。')

## 结果检查与归档

先检查当前 run 目录，再只打包这一次运行的结果，方便之后下载。

In [ ]:
!ls -lah /content/Surper_GCG/results/pipeline_runs
!ls -lah /content/Surper_GCG/results/pipeline_runs/$ACTIVE_RUN_NAME

In [ ]:
!zip -r /content/drive/MyDrive/${ACTIVE_RUN_NAME}_pipeline_results.zip /content/Surper_GCG/results/pipeline_runs/$ACTIVE_RUN_NAME